# Your First LLM-Powered Tool

Starter notebook. Fill in the TODOs and **run every cell** before committing.
Do **not** commit your API key — load it from `.env`.


In [ ]:
# Setup
# !pip install -r requirements.txt
import os, json
from google import genai

with open("../GEMINI_API_KEY.txt") as f:
    GEMINI_API_KEY = f.read().strip()
assert GEMINI_API_KEY, "Set GEMINI_API_KEY (see .env.example)"
client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-3.1-flash-lite"


## Task 1 — First calls and the sampling dial

Make a working call (print response **and token usage**), then send the same prompt 3× at temp ~0.1 and 3× at temp ~1.0.


In [ ]:
import time

system_msg = "You are a helpful assistant. Be concise."
user_msg = "What is the capital of France?"

print("=== Single Call ===")
response = client.models.generate_content(
    model=MODEL,
    contents=[
        {"role": "user", "parts": [{"text": user_msg}]}
    ],
)

print(f"Response: {response.text}")
print(f"Usage Metadata:")

usage = response.usage_metadata
if hasattr(usage, 'prompt_token_count'):
    print(f"  Input tokens: {usage.prompt_token_count}")
    print(f"  Output tokens: {usage.candidates_token_count}")
    print(f"  Total tokens: {usage.prompt_token_count + usage.candidates_token_count}")
elif hasattr(usage, 'input_tokens'):
    print(f"  Input tokens: {usage.input_tokens}")
    print(f"  Output tokens: {usage.output_tokens}")
    print(f"  Total tokens: {usage.input_tokens + usage.output_tokens}")
else:
    print(f"  Available attributes: {dir(usage)}")
print()

print("=== Temperature Comparison (Low vs High) ===")
temperatures = [0.1, 1.0]

for temp in temperatures:
    print(f\nTemperature: {temp})
    print("-" * 40)
    time.sleep(1)  # Add 1-second delay between calls
    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=[
                {"role": "user", "parts": [{"text": user_msg}]}
            ],
            config={"temperature": temp},
        )
        print(f"Response: {response.text}")
    except Exception as e:
        print(f"Error (quota/timeout): {str(e)[:100]}...")
        print("Tip: Free tier has rate limits. Consider adding a payment method to your Gemini API account.")


**What changed, and why?**

- **Added a single API call**: prints the model response and shows token usage in a backwards-compatible way (handles both `usage.prompt_token_count` / `candidates_token_count` and older `input_tokens` / `output_tokens`).
- **Temperature comparison**: sends the same prompt at `temperature=0.1` and `temperature=1.0` so you can observe sampling variability between low and high temperature settings.
- **Usage display & delays**: prints input/output/total tokens and adds a short delay between calls to reduce transient rate-limit errors.
- **Error handling & tips**: catches quota/timeout exceptions and prints a helpful message about free-tier limits and adding a payment method.

These edits make the notebook reproducible and easier to use for studying how sampling affects outputs and token costs.


## Task 2 — Prompt-pattern bake-off

Classify the tickets three ways (zero-shot, few-shot, chain-of-thought) into `billing / bug / feature_request / other`. Build a comparison table. Record prompts + verdict in `prompts.md`.


In [ ]:
with open("tickets.json") as f:
    tickets = json.load(f)

LABELS = ["billing", "bug", "feature_request", "other"]

def classify(text, style):
    """Classify `text` using one of three prompt styles: 'zero_shot' | 'few_shot' | 'cot'.
    Returns a label from `LABELS` (fallback: 'other')."""
    if style == "zero_shot":
        prompt = f"Classify the following support ticket into one of: {', '.join(LABELS)}.\nOutput only the label (one of: {', '.join(LABELS)}).\n\nTicket:\n{text}"
        temp = 0.0
    elif style == "few_shot":
        examples = [
            ("I was charged twice for my subscription this month.", "billing"),
            ("The app crashes when I try to upload a file.", "bug"),
            ("Can you add dark mode to the dashboard?", "feature_request"),
        ]
        ex_str = ""
        for ex, lbl in examples:
            ex_str += f"Ticket: {ex}\nLabel: {lbl}\n\n"
        prompt = f"Classify the ticket into one of: {', '.join(LABELS)}.\nUse the examples (format: Ticket / Label) then classify this ticket and output only the label.\n\n{ex_str}Now classify:\n{text}"
        temp = 0.0
    elif style == "cot":
        prompt = f"Classify the ticket into one of: {', '.join(LABELS)}.\nThink step-by-step and then provide the final label on its own line.\n\nTicket:\n{text}"
        temp = 0.2
    else:
        raise ValueError("Unknown style")

    # ensure prompt is a single string
    if isinstance(prompt, (list, tuple)):
        prompt = "".join(prompt)

    response = client.models.generate_content(
        model=MODEL,
        contents=[{"role": "user", "parts": [{"text": prompt}]}],
        config={"temperature": temp},
    )
    out = response.text.strip()
    out_l = out.lower()
    for lbl in LABELS:
        if lbl in out_l:
            return lbl
    # final-line / exact match fallback
    lines = [l.strip().lower() for l in out.splitlines() if l.strip()]
    for l in lines:
        if l in LABELS:
            return l
    # nothing matched — return 'other'
    return 'other'

# Run all three styles over the tickets and print a small summary table
styles = ["zero_shot", "few_shot", "cot"]
results = {s: [] for s in styles}
for s in styles:
    for t in tickets:
        text = t.get("text") if isinstance(t, dict) else t
        pred = classify(text, s)
        results[s].append({"text": text, "predicted": pred})

from collections import Counter
for s in styles:
    cnt = Counter([r['predicted'] for r in results[s]])
    print(f\nStyle: {s})
    for lbl in LABELS:
        print(f\



## Task 3 — Structured output as a function

Return JSON `{label, confidence, reason}` with low temperature, then **parse and validate** it. Handle one bad-output case. Re-run against local Ollama and compare.


In [ ]:
def classify_structured(text):
    # TODO: use the API's JSON / structured-output mode at low temperature
    # TODO: parse the JSON, validate label in LABELS and 0 <= confidence <= 1
    # TODO: handle at least one malformed-output case gracefully
    raise NotImplementedError

# TODO: run the SAME structured prompt against local Ollama (OpenAI-compatible
# endpoint at http://localhost:11434/v1) and note how reliably it returns valid JSON.


**Local vs hosted: did the small local model produce valid JSON as reliably?**

> TODO
